<a href="https://colab.research.google.com/github/vorobyevakatya/Comp-Linguistics/blob/main/fine_tuning_hw.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#### Домашнее задание

**Датасет:** [`ag_news`](https://huggingface.co/datasets/fancyzhx/ag_news) — классификация новостей по 4-м категориям (World, Sports, Business, Sci/Tech)

**Техническое задание:**

1.  Загрузите датасет `ag_news`
2.  Выберите модель для дообучения (например, `distilbert-base-uncased` или `bert-base-uncased`), `num_labels=4`
3.  Токенизируйте данные (`max_length=128`)
4.  Настройте `TrainingArguments`:
    *   `learning_rate = 2e-5`
    *   `per_device_train_batch_size = 16`
    *   `num_train_epochs = 3`
    *   `eval_strategy = "epoch"`
    *   `save_strategy = "epoch"`
    *   `load_best_model_at_end = True`
    *   `metric_for_best_model = "accuracy"`
5.  Обучите модель с помощью `Trainer`. Для метрик используйте `evaluate.load("accuracy")`
6.  Выведите accuracy на тестовой выборке
7.  Сохраните модель в папку `./ag_news_model`
8.  Протестируйте модель на трех новых новостях (вписать вручную), используя `pipeline`. Выведите предсказанный класс и уверенность модели

In [2]:
!pip install evaluate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 6.5 MB/s eta 0:00:00


In [3]:
import numpy as np

from datasets import load_dataset
ds = load_dataset("fancyzhx/ag_news")

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments,
    pipeline
)
import evaluate
import torch

print("Загружаем датасет AG News")
dataset = load_dataset("ag_news")

print(f"Структура датасета: {dataset}")
print(f"Пример новости: {dataset['train'][0]}")
print(f"Классы: {dataset['train'].features['label'].names}")

model_name = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)

def tokenize_function(examples):
    return tokenizer(
        examples['text'],
        padding="max_length",
        truncation=True,
        max_length=128
    )
print("Токенизируем данные")
tokenized_datasets = dataset.map(tokenize_function, batched=True)

train_dataset = tokenized_datasets["train"]
test_dataset = tokenized_datasets["test"]

model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=4
)

training_args = TrainingArguments(
    output_dir="./ag_news_results",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    logging_dir='./logs',
    logging_steps=500,
)

accuracy_metric = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=1)
    return accuracy_metric.compute(predictions=predictions, references=labels)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    compute_metrics=compute_metrics,
)

print("Начинаем обучение модели")
trainer.train()

print("Оцениваем модель на тестовой выборке")
test_results = trainer.evaluate()
print(f"Accuracy на тестовой выборке: {test_results['eval_accuracy']:.4f}")

model_save_path = "./ag_news_model"
trainer.save_model(model_save_path)
tokenizer.save_pretrained(model_save_path)
print(f"Модель сохранена в {model_save_path}")

print("\n" + "="*50)
print("ТЕСТИРОВАНИЕ НА НОВЫХ НОВОСТЯХ")
print("="*50)

classifier = pipeline(
    "text-classification",
    model=model_save_path,
    tokenizer=model_save_path
)

id2label = {
    'LABEL_0': 'World',
    'LABEL_1': 'Sports',
    'LABEL_2': 'Business',
    'LABEL_3': 'Sci/Tech'
}

test_news = [
    "Argentina defeated France in the 2022 World Cup final. Lionel Messi scored twice and was named tournament's best player. The match ended 3-3 after extra time, Argentina won 4-2 on penalties.",

    "Roscosmos successfully launched a Soyuz-2.1b rocket from Vostochny Cosmodrome. The rocket delivered a Meteor-M weather satellite and 42 smaller spacecraft into orbit. This was the 10th launch from Vostochny in 2024.",

    "Milan Fashion Week opened with Gucci's new collection showcase. The creative director presented clothing made from recycled materials. Over 50 fashion houses from Italy, France and Britain are participating."
]

for i, news in enumerate(test_news, 1):
    print(f"\nНовость {i}:")
    print(f"Текст: {news[:100]}...")

    result = classifier(news)[0]
    label = result['label']
    score = result['score']

    category = id2label.get(label, label)
    print(f"Предсказанная категория: {category}")
    print(f"Уверенность модели: {score:.4f}")

print("\n" + "="*50)
print("задание выполнено")
print("="*50)

Загружаем датасет AG News


README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/18.6M [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/1.23M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/120000 [00:00<?, ? examples/s]

...

Accuracy на тестовой выборке: 0.9480


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Модель сохранена в ./ag_news_model

ТЕСТИРОВАНИЕ НА НОВЫХ НОВОСТЯХ


Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]


Новость 1:
Текст: Argentina defeated France in the 2022 World Cup final. Lionel Messi scored twice and was named tourn...
Предсказанная категория: Sports
Уверенность модели: 0.9943

Новость 2:
Текст: Roscosmos successfully launched a Soyuz-2.1b rocket from Vostochny Cosmodrome. The rocket delivered ...
Предсказанная категория: Sci/Tech
Уверенность модели: 0.9970

Новость 3:
Текст: Milan Fashion Week opened with Gucci's new collection showcase. The creative director presented clot...
Предсказанная категория: Business
Уверенность модели: 0.6025

задание выполнено
